# The Physics of Heat Conduction
Heat conduction is the process of thermal energy transfer through a material without any macroscopic movement of the material itself. It occurs due to microscopic interactions between particles, such as collisions of molecules in gases or vibrations of atoms in solids.
## Mechanisms of Heat Transfer
There are three fundamental ways that heat can bye transferred:
1. Radiation - transfer of heat via electrodynamic waves
2. Convection - transfer of heat via bulk movement of fluids
3. **Conduction** - transfer of heat through material without bulk motion

For conduction, thermal energy flows from higher to lower temperature regions caused by the thermal agitation of particles. Heat flows are determined by temperature variations through the material or gas. As with any physical problem like this, an initial temperature profile must be specificed before predicting the evolution of the system.

## Fourier's Law of Heat Conduction
The fundamental law governing heat conduction is Fourier's Law, which states that the rate of heat transfer (heat flux) through a material is proportional to the negative temperature gradient: $$\vec{Q} = -k\nabla T,$$ where $Q$ is the heat flux (a vector quantity describing the rate and direction of heat flow, in $W/m^2$) and $k$ is the thermal conductivity of the material ($W/m\cdot K$). Essential, heat flows at right angles to the temperature isotherms, where the negative sign ensures that heat flows from high to low temperatures. This is not unlike how electric field lines are perpendicular to equipotential surfaces, giving a relation of $\vec{E}=-\nabla V$.

At any instant of time the heat flow through an object equals the variation of energy inside the material, $$\mathrm{Heat\,\,Flow=Variation\,\,of\,\,internal\,\,energy}$$
or 
$$\nabla \cdot Q = -\frac{dE}{dt}.$$
The variation of the internal energy is given by the body's ability to store heat by raising its temperature, $$\frac{dE}{dt} = \rho c \frac{dT}{dt},$$ 
where $\rho$ is the density, and $c$ is the specific heat of the material. We can use Fourier's Law to equate temperature variations, 
$$\nabla \cdot \left( k \nabla T\right) = \rho c \frac{dT}{dt},$$ 
$$\frac{dT}{dt}=\alpha \nabla^2T,$$ 
with $\alpha = k/(c\rho)$. In one dimension, this equation is written
$$\frac{\partial T(x,t)}{\partial t} = \alpha \frac{\partial ^2T}{\partial x^2}.$$ 

We must solve this equation given the initial condition 
$$T(x,t=0) = f(x)\,\,\,\mathrm{(initial\,\,condition)}$$ 
and the boundary condition 
$$\begin{align}
T(x=0,t) &=& T_1, \\
T(x=l,t) &=& T_2.
\end{align}$$

# An Example: A bar with the ends in contact with heat reservoirs

![bar](figures/bar.png)

Let's model the heat flux through an iron bar that is in contact with heat reservoirs at either end. The model follows familiar steps of discretizing the difference method and implementing FDM.


## Finite differences solution


![heat](figures/heat.png)

#### The finite differences algorithm for the heat equation.


The numerical solution is based on converting the differential equation into an approximate finite-diffence one. Following a derivation similar to the one we used for the wave equation we approximate the derivatives by finite differences:
$$\begin{align}
\frac{\partial T}{\partial t} &=& \frac{T(x,t+\Delta t)-T(x,t)}{\Delta t}, \\
\frac{\partial^2 T}{\partial x^2} &=& \frac{T(x+\Delta x,t)+T(x-\Delta x,t)-
2T(x,t)}{(\Delta x)^2}.\end{align}$$

Substituting and rearranging, we obtain the discrete equivalent:
$$T(x,t+\Delta t)=T(x,t)+\frac{\alpha}{C}\left[T(x+\Delta x,t)+T(x-\Delta x,t) -2T(x,t) \right], $$ with the constant $C=(\Delta x)^2/\Delta t$. We see in the figure above that the temperature at the point $(x, t+\Delta t)$ is determined by the temperatures at three points of the previous time step. The boundary conditions impose fixed values along the perimeter. An initial condition is used to generate the temperature gradient at time $t=\Delta t$, and the discrete equation is used for the time evolution.

The stability condition for a numerical solution is given by

$$\alpha\frac{\Delta t}{(\Delta x)^2} \leq \frac{1}{2d}.$$ 
where $d$ represents the dimension of the problem

This means that if we make the time step smaller we improve convergence, but if we decrease the space step without a simultaneous quadratic increase of the time step, we worsen it. The stability condition can be found by using the substituting in the form $T_{j}^n = A^n e^{i kj\Delta x}$, ensuring that the amplification factor $|A|<1$.

### Finite differences program 

Solve the temperature distribution within an iron bar of length $l=50$ cm with the boundary conditions $$T(x=0,t)=T(x=l,t)=0,$$ and initial conditions $$T(x,t=0)=100^{\circ}\mathrm{C}.$$ The corresponding constants for iron are:
$$c=0.113\, \mathrm{cal/(g^{\circ} C)}$$ $$ k=0.12\, \mathrm{cal/(cm\,s^{\circ}C)}$$ $$\rho=7.8 \,\mathrm{g/cm^3}.$$





In [ ]:
%matplotlib inline
import numpy as np
from matplotlib import pyplot
import matplotlib.animation as animation
from IPython.display import HTML

# Properties for iron
k = 0.12 #(cm s K)
c = 0.113 #(cal/g K)
rho = 7.8 #(g/cm3)
alpha = k/(c*rho)

l = 5    #length of the bar
dx = 0.125  # space step
nx = int(l/dx)  # number of points in space
x = np.arange(0,l+dx,dx) # the +1 is necessary to store the value at l
dt = 0.01
C = dx**2/dt
r = alpha/C
#Check Stability!
assert r<1/2, 'Stability conditions not met!'

# these arrays will contain the new displacements at t, and t+delta
t0 = np.zeros(x.shape)
t1 = np.zeros(x.shape) 

#Initial conditions
t0[:] = 100.
t0[0] = 0.
t0[nx] = 0.

fig = pyplot.figure()
ax = pyplot.axes(xlim=(0, l), ylim=(0, 120), xlabel='x', ylabel='T')
points, = ax.plot([], [], marker='', linestyle='-', lw=3)

def evolve(i):
    #globalize t0 and t1
    global t0, t1

    for ix in range(1,nx):
        t1[ix] = t0[ix] + r*(t0[ix+1]+t0[ix-1]-2*t0[ix])  

    points.set_data(x, t0)
    #store t0->t1
    for ix in range(nx):
        t0[ix] = t1[ix]

    return (points,)

anim = animation.FuncAnimation(fig, evolve, frames = 2000, interval=10)

HTML(anim.to_jshtml())            

# Example 2: Two bars in contact 

Let's consider two identical bars, 25cm long each, are in thermal contact. One bar is initially at $50^{\circ}C$, and the other at $100^{\circ}C$. The free ends are kept at $0^{\circ}C$. Calculate and plot the temperature distribution as a function of time. What needs to change?

In [ ]:
%matplotlib inline
import numpy as np
from matplotlib import pyplot
import matplotlib.animation as animation
from IPython.display import HTML

#iron properties
k = 0.12 
c = 0.113
rho = 7.8
alpha = k / (c * rho)

l = 5.0          # total length of both bars (cm)
dx = 0.125       # space step
nx = int(l / dx) # number of interior + boundary points - 1
x = np.arange(0, l + dx, dx)  # nx+1 points: 0 to l inclusive

dt = 0.01
C = dx**2 / dt
r = alpha / C

# Check stability
assert r < 1/2, f'Stability condition not met! r = {r:.4f}'

# Temperature arrays
t0 = np.zeros(x.shape)
t1 = np.zeros(x.shape)

# Initial conditions: left bar at 50°C, right bar at 100°C
# Free ends held at 0°C (Dirichlet BCs)
t0[1:int(nx/2)] = 50.
t0[int(nx/2):nx] = 100.
#Heat sinks at boundaries
t0[0]  = 0.   # left boundary
t0[nx] = 0.   # right boundary

fig = pyplot.figure()
ax = pyplot.axes(xlim=(0, l), ylim=(0, 120), xlabel='x (cm)', ylabel='T (°C)')
points, = ax.plot([], [], marker='', linestyle='-', lw=3)

def evolve(i):
    global t0, t1

    # Interior points only; boundaries stay fixed
    t1[1:nx] = t0[1:nx] + r * (t0[2:nx+1] + t0[0:nx-1] - 2*t0[1:nx])

    # Boundaries remain at 0°C (already zero in t1, but explicit is better)
    t1[0]  = 0.
    t1[nx] = 0

    points.set_data(x, t1)
    t0[:] = t1[:]

    return (points,)

anim = animation.FuncAnimation(fig, evolve, frames=2000, interval=10)
HTML(anim.to_jshtml())

## Heat Conduction on a 2D sheet
What changes when considering heating conduction in 2 dimensions.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

# Material properties for iron
k = 0.12    # Thermal conductivity (cal/ s cm K)
c = 0.113   # Specific heat capacity (cal/g K)
rho = 7.8   # Density (g/cm3)
alpha = k / (c * rho)  # Thermal diffusivity

# Grid setup
Lx = 
Ly = 

# Initialize temperature field


def evolve(T):
    """Computes the next time step of heat propagation."""
    

# Set up the figure for animation
fig, ax = plt.subplots()
cmap = ax.imshow(temp, cmap='hot', interpolation='nearest', origin='lower', extent=[0, Lx, 0, Ly])
ax.set_xlabel("x (cm)")
ax.set_ylabel("y (cm)")
ax.set_title("Heat Propagation in 2D Iron Plate")
fig.colorbar(cmap, label="Temperature (C)")

def animate(frame):
    global temp
    temp = evolve(temp)
    cmap.set_array(temp)
    return [cmap]

# Run the animation
ani = animation.FuncAnimation(fig, animate, frames=1000, interval=10, blit=False)

HTML(ani.to_jshtml())   